In [51]:
from qdrant_client import QdrantClient
from pprint import pprint
from qdrant_client import models
from qdrant_client.models import PointStruct
from fastembed import SparseTextEmbedding

client = QdrantClient(url="http://localhost:6333")

collection_name = "one_paper"
embedding_model_dimensions = 384

dense_model = "BAAI/bge-small-en"
sparse_model = "qdrant/bm25"

In [52]:
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense_vector": models.VectorParams(
                size=embedding_model_dimensions, distance=models.Distance.COSINE
            )
        },
        sparse_vectors_config={
            "bm25_sparse_vector": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            )
        },
    )

In [ ]:
# client.delete_collection(collection_name=collection_name)

True

In [ ]:
from sentence_transformers import SentenceTransformer

sentences = [
    "The patient was administered 500mg of ibuprofen post-surgery.",
    "After the operation, the doctor gave the patient pain relief medication.",
    "LightGBM uses gradient boosting on decision trees to minimize a loss function iteratively.",
    "The model trains by correcting its own mistakes in successive rounds.",
    "The quarterly revenue for Q3 2024 was NPR 4,200,000.",
    "Sales went up significantly in the third quarter of last year.",
    "The API returns a 429 status code when the rate limit is exceeded.",
    "Too many requests were sent in a short period and the server rejected them.",
    "She left the company in March 2023 after the acquisition was finalized.",
    "The employee resigned shortly after the merger deal closed.",
    "SHAP values decompose a model's prediction into per-feature contributions.",
    "You can see exactly how much each variable pushed the output up or down.",
    "The contract was governed under Section 47(b) of the Nepal Civil Code 2074.",
    "The agreement was subject to local civil law provisions.",
    "PostgreSQL's MVCC ensures readers don't block writers during concurrent transactions.",
    "The database handles simultaneous reads and writes without locking.",
    "The patient's HbA1c was 8.2%, indicating poorly controlled diabetes.",
    "The blood test results showed the sugar levels were too high over time.",
    "Invoice #INV-2024-0391 was issued on 14th September for USD 3,800.",
    "A bill was sent in mid-September for consulting services rendered.",
]

dense_encoder = SentenceTransformer(dense_model)
dense_embeddings = dense_encoder.encode(sentences)

bm25_encoder = SparseTextEmbedding(model_name=sparse_model)
sparse_embeddings = list(bm25_encoder.embed(sentences))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7583.26it/s]


In [75]:
sparse_embeddings[0]

SparseEmbedding(values=array([1.68774348]), indices=array([851380037]))

In [76]:
points = []
for idx, (item, dense_vec, sparse_vec) in enumerate(
    zip(sentences, dense_embeddings, sparse_embeddings)
):
    point = PointStruct(
        id=idx + 1,
        payload={"sentence": item},
        vector={
            "dense_vector": dense_vec.tolist(),
            "bm25_sparse_vector": models.SparseVector(
                indices=sparse_vec.indices.tolist(), values=sparse_vec.values.tolist()
            ),
        },
    )
    points.append(point)

client.upload_points(collection_name=collection_name, points=points, batch_size=8)

In [99]:
query = "MVCC"

dense_query_vec = dense_encoder.encode(query)
sparse_query_vec = next(bm25_encoder.embed([query]))

In [100]:
results = client.query_points(
    collection_name=collection_name,
    prefetch=[
        models.Prefetch(query=dense_query_vec.tolist(), using="dense_vector", limit=5),
        models.Prefetch(
            query=models.SparseVector(
                indices=sparse_query_vec.indices.tolist(),
                values=sparse_query_vec.values.tolist(),
            ),
            using="bm25_sparse_vector",
            limit=5,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=5,
)


pprint(results.points)

[ScoredPoint(id=15, version=2, score=1.0, payload={'sentence': "PostgreSQL's MVCC ensures readers don't block writers during concurrent transactions."}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1, version=4, score=0.33333334, payload={'sentence': 'sampada'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=11, version=2, score=0.25, payload={'sentence': "SHAP values decompose a model's prediction into per-feature contributions."}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=1, score=0.2, payload={'sentence': 'LightGBM uses gradient boosting on decision trees to minimize a loss function iteratively.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=19, version=3, score=0.16666667, payload={'sentence': 'Invoice #INV-2024-0391 was issued on 14th September for USD 3,800.'}, vector=None, shard_key=None, order_value=None)]
